# Tratado de datos — Plan de Capacitación Cobranza

Convierte las fuentes crudas mensuales de Cobranza en:

1. Las dos salidas finales en **CSV**, para lectura humana (**Detalle Colaborador** y
   **Avance por curso, región y centro**).
2. Un paquete **`.macintosh.json`** (sección 10) para alimentar directamente al
   **tablero de Cobranza** (Apps Script, carpeta `src/` del repo) vía su botón
   "Actualizar datos" — mismos números, sin volver a tocar nada a mano.

Reemplaza a `Tuberia/Cobranza.sql` (DuckDB), que quedó desactualizado porque:

- El `Plan de Capacitación Cobranza.xlsx` ahora trae **4 pestañas** (cursos/colaboradores
  generales + cursos/colaboradores específicos), no una sola hoja `Hoja1`.
- Hay que traducir la región cruda (Tienda/RRHH) a las **15 regiones oficiales de
  Cobranza**, cruzando contra la pestaña `CENTROS-TIPOCENTROS` de la Planta por
  Posiciones (la "Plantilla de Cobranza").
- Ahora sí se necesita la columna **Centro** en ambas salidas (en el ejemplo que me
  diste no aparece porque esa área no la usa, pero Cobranza sí).
- Hay **dos** planes de capacitación, no uno: el de Colaborador (el original) y el
  **Gerencial** (mismas 4 pestañas, para puestos de liderazgo). Ambos se unen — ver
  supuesto #6.

## ⚠️ Supuestos que debes confirmar al correrlo (revisa la sección 7 · Validaciones)

1. **Estructura del plan**: asumo que `Plan de Capacitación Cobranza.xlsx` tiene las
   pestañas `Cursos asignados`, `Colaboradores asignados`, `Cursos específicos` y
   `Colaboradores específicos` tal como describe `Estructura de origen.txt`. Si el
   archivo real trae otros nombres de pestaña/columna, ajusta la sección 1.
2. **"Rango de meses para cursar"** (ej. `0-3`): tomo el número **mínimo** del rango
   como el punto a partir del cual el curso aplica (`dias_laborados >= mínimo*30`),
   igual de estricto que el SQL viejo pero generalizado a un rango. Si la regla real
   es distinta (por ejemplo, que el curso deja de aplicar después del máximo), dímelo
   y lo ajusto.
3. **Cruce de finalizaciones**: uno el plan contra `Cobranza_P*.csv` por el nombre
   exacto del curso (columna `Curso` del plan == `Nombre Curso` del csv). Si el
   nombre no calza exactamente entre ambos archivos, el curso queda siempre como
   pendiente; la sección 7 imprime los cursos del plan que nunca encontraron ninguna
   coincidencia para que lo detectes rápido.
4. **Excepción de "Formación de Conductores"** (puestos 743/721 en los 12 centros de
   Impresión de Cobranza): no la programo a mano — confío en que la pestaña
   `Colaboradores específicos` ya la tenga bien reflejada (por eso existe esa pestaña).
   La sección 7 sí trae una validación que avisa si algún colaborador con puesto
   743/721 en esos centros terminó recibiendo la especialización, por si la fuente no
   la excluyó.
5. **"Centros de costos"** en `Colaboradores específicos`: si una fila trae varios
   centros en una sola celda (separados por coma, `;` o salto de línea), los separo
   automáticamente. Si cada centro ya viene en su propia fila, no pasa nada (el
   separador simplemente no encuentra nada que partir).
6. **Plan Colaborador + Plan Gerencial se suman**: confirmado — un puesto como 721
   (Gerente de Operación Cobranza) o 743 (Jefe de Operación Cobranza), que aparece en
   los dos planes, recibe los cursos de **ambos** (no se reemplazan entre sí). La
   sección 7 trae una validación (`personas_ambos_planes`) para que puedas ver
   exactamente a quién le está aplicando esta regla.
7. **Formato de los PDT (Colaborador/Operación y Gerencial)**: cada uno puede llegar
   como 4 archivos `.csv` sueltos (uno por pestaña, p. ej.
   `PDT-operacion-adaptado-...-Cursos_asignados.csv` o
   `PDT-gerencial-adaptado-...-Cursos_asignados.csv`) o como un solo `.xlsx` con esas
   4 pestañas — `encontrar_plan_pdt()` prueba primero los CSV sueltos y cae al `.xlsx`
   si no los encuentra completos. Ya no asumo el nombre viejo
   "Plan de Capacitación Cobranza.xlsx": ambos planes se buscan por el alias que trae
   cada archivo en su nombre (`operaci[oó]n` / `gerencial`), sin importar guiones,
   fechas u otro texto alrededor.

## Qué SÍ cambia solo cada mes (nada que editar a mano más allá de la sección 0)

- **Carpeta de Drive, etiqueta de los CSV y periodo del paquete**: los tres salen de
  `AÑO`/`MES` en la sección 0 — no hay que retipear "Agosto 2026" en tres lugares
  distintos ni arriesgarse a que queden desincronizados.
- **Nombres de archivo** (sufijos tipo `(12)`, rangos de fecha tipo
  `06Ene_01Ago2026`): se buscan por patrón, no por nombre exacto. Si hay más de un
  archivo que combina, se usa el modificado más recientemente y se avisa.
- **Pestañas de fecha** (`01 AGO 26`, etc.) en Planta por Posiciones/Centro: se elige
  automáticamente la más reciente, Y se avisa fuerte si esa pestaña no corresponde al
  `AÑO`/`MES` que estás procesando (señal de que esa fuente no se ha actualizado con
  el corte de este mes).
- **Estructura mixta de Abril/Junio/Julio** en Planta por Posiciones (una tabla
  resumen encima de la tabla nominal real, según `Estructura de origen.txt`): se
  detecta sola buscando la fila donde de verdad empiezan los encabezados
  (`Número de trabajador`), en vez de asumir que el encabezado siempre es la fila 0.
- **Cuántos periodos `Cobranza_P*.csv` haya** (a veces 1, a veces varios): se
  concentran todos los que encuentre, sin límite fijo.
- **Mayúsculas/minúsculas y espacios sobrantes en nombres de archivo/columna**: la
  búsqueda de archivos ignora mayúsculas, y `limpiar_columnas()` quita espacios de más
  en los encabezados (el CSV real del plan Gerencial trae la columna `"Puesto "` con
  un espacio al final; sin esto, un `KeyError` invisible habría tronado el pipeline).

Corre las celdas en orden. Todo lo que necesitas ajustar (año, mes, y la fecha de
corte si no fuera fin de mes) está en la sección **0 · Configuración**.


## 0 · Preparación y configuración

In [1]:
# Solo hace falta la primera vez por sesión de Colab.
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import fnmatch
import glob
import json
import os
import re
import unicodedata
from datetime import date

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)


In [3]:
# ============================================================================
# CONFIGURACIÓN — lo ÚNICO que hay que tocar cada mes son estas 3 líneas.
# Todo lo demás (nombre de la carpeta de Drive, etiqueta de los CSV, periodo del
# paquete del tablero) se deriva de aquí, para que no puedan quedar desincronizados
# entre sí por un dato que se te haya olvidado actualizar en algún lugar.
# ============================================================================

AÑO = 2026
MES = 8  # 1 = Enero ... 12 = Diciembre

# La fecha de corte casi siempre es fin de mes, pero no siempre (a veces el área
# cierra antes). Se calcula fin de mes por defecto; sobreescribe esta línea si el
# corte real de este mes fue otro día.
import calendar
FECHA_CORTE = f"{AÑO:04d}-{MES:02d}-{calendar.monthrange(AÑO, MES)[1]:02d}"

MESES_ES = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
    7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre',
}
CARPETA_MES = f"{MESES_ES[MES]} {AÑO}"   # ej. 'Agosto 2026' -> nombre real de la carpeta en Drive
ETIQUETA_MES = CARPETA_MES               # se reutiliza para nombrar los CSV/paquete de salida

RUTA_BASE = f"/content/drive/MyDrive/Automatizaciones-GC/{CARPETA_MES}/Cobranza"
RUTA_CRUDOS = f"{RUTA_BASE}/Datos crudos"
RUTA_RESULTADOS = f"{RUTA_BASE}/Resultados"

print(f"Procesando: {CARPETA_MES}  |  Fecha de corte: {FECHA_CORTE}")
print(f"Carpeta de origen: {RUTA_CRUDOS}")

if not os.path.isdir(RUTA_CRUDOS):
    hermanas = os.listdir(os.path.dirname(RUTA_CRUDOS)) if os.path.isdir(os.path.dirname(RUTA_CRUDOS)) else []
    raise FileNotFoundError(
        f"No existe la carpeta '{RUTA_CRUDOS}'. "
        f"¿AÑO/MES ({AÑO}/{MES}) apuntan al mes correcto? "
        f"Carpetas que sí existen en '{os.path.dirname(RUTA_CRUDOS)}': {hermanas or '(ninguna, revisa RUTA_BASE)'}"
    )

REGIONES_COBRANZA_OFICIALES = [
    'TORREON', 'CULIACAN', 'HERMOSILLO', 'MEXICALI', 'LEON', 'MONTERREY',
    'TOLUCA', 'GUADALAJARA', 'QUERETARO', 'CUAUTITLAN IZCALLI', 'PUEBLA',
    'VERACRUZ', 'IXTAPALUCA', 'VILLAHERMOSA', 'MERIDA',
]

# Puestos y centros de la excepción "Centro de Impresión de Cobranza"
# (Lógicas Tableros Cobranza, página 2). Solo se usan para VALIDAR, no para decidir.
PUESTOS_EXCEPCION_IMPRESION = {'743', '721'}
CENTROS_IMPRESION_COBRANZA = {
    '500306', '521902', '504704', '501004', '503605', '509104',
    '501105', '504307', '507703', '509505', '504804', '517002',
}


Procesando: Agosto 2026  |  Fecha de corte: 2026-08-31
Carpeta de origen: /content/drive/MyDrive/Automatizaciones-GC/Agosto 2026/Cobranza/Datos crudos


## 1 · Utilidades

Funciones que reemplazan las macros de DuckDB (`fecha_excel`, `texto_clave`) y que
resuelven los nombres de archivo/pestaña que cambian cada mes.

In [4]:
def buscar_archivos_ci(carpeta, patron):
    """Como glob, pero insensible a mayúsculas/minúsculas (Drive no siempre
    respeta el caso tal cual se escribió el nombre original)."""
    if not os.path.isdir(carpeta):
        return []
    patron_low = patron.lower()
    return sorted(
        os.path.join(carpeta, nombre) for nombre in os.listdir(carpeta)
        if fnmatch.fnmatch(nombre.lower(), patron_low)
    )


def encontrar_archivo(carpeta, patron):
    """Busca un archivo por patrón dentro de la carpeta de datos crudos (sin
    importar mayúsculas/minúsculas). Si hay varios que combinan, usa el
    modificado más recientemente y avisa."""
    candidatos = buscar_archivos_ci(carpeta, patron)
    if not candidatos:
        raise FileNotFoundError(f"No se encontró ningún archivo '{patron}' en {carpeta}")
    if len(candidatos) > 1:
        candidatos.sort(key=os.path.getmtime, reverse=True)
        print(f"Aviso: varios archivos combinan con '{patron}'; se usa el más reciente: "
              f"{os.path.basename(candidatos[0])}")
    return candidatos[0]


def encontrar_archivo_opcional(carpeta, patron):
    """Como encontrar_archivo, pero regresa None en vez de tronar si no hay nada."""
    candidatos = buscar_archivos_ci(carpeta, patron)
    if not candidatos:
        return None
    if len(candidatos) > 1:
        candidatos.sort(key=os.path.getmtime, reverse=True)
        print(f"Aviso: varios archivos combinan con '{patron}'; se usa el más reciente: "
              f"{os.path.basename(candidatos[0])}")
    return candidatos[0]


def encontrar_archivos_periodos(carpeta):
    """Encuentra todos los Cobranza_P1.csv, Cobranza_P2.csv, ... sin importar cuántos haya."""
    patrones = ["Cobranza_P*.csv", "Cobranza P*.csv", "Cobranza*P[0-9]*.csv"]
    vistos, archivos = set(), []
    for patron in patrones:
        for ruta in buscar_archivos_ci(carpeta, patron):
            if ruta not in vistos:
                vistos.add(ruta)
                archivos.append(ruta)
    if not archivos:
        raise FileNotFoundError(f"No se encontraron archivos Cobranza_P*.csv en {carpeta}")
    return sorted(archivos)


def limpiar_columnas(df):
    """Quita espacios sobrantes en los nombres de columna (p. ej. 'Puesto ' con
    espacio al final, como trae el CSV real del plan gerencial) para que no
    truene un KeyError por un espacio invisible."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


_MESES_ABBR = {
    'ENE': 1, 'FEB': 2, 'MAR': 3, 'ABR': 4, 'MAY': 5, 'JUN': 6,
    'JUL': 7, 'AGO': 8, 'SEP': 9, 'OCT': 10, 'NOV': 11, 'DIC': 12,
}


def _parsear_fecha_pestana(nombre):
    partes = nombre.strip().upper().split()
    if len(partes) != 3:
        return None
    dia, mes_abbr, anio = partes
    mes = _MESES_ABBR.get(mes_abbr[:3])
    if mes is None or not dia.isdigit() or not anio.isdigit():
        return None
    anio_completo = 2000 + int(anio) if len(anio) == 2 else int(anio)
    try:
        return date(anio_completo, mes, int(dia))
    except ValueError:
        return None


def hoja_mas_reciente(ruta_excel, anio_esperado=None, mes_esperado=None):
    """Entre pestañas tipo '01 AGO 26', regresa el nombre de la más reciente.

    Si se pasan anio_esperado/mes_esperado (el AÑO/MES que se está procesando) y
    la pestaña más reciente NO es de ese mes, avisa fuerte: probablemente el área
    dueña del archivo todavía no sube el corte de este mes y el pipeline seguiría
    corriendo en silencio con datos del mes anterior si no se revisa."""
    xls = pd.ExcelFile(ruta_excel)
    con_fecha = [(hoja, _parsear_fecha_pestana(hoja)) for hoja in xls.sheet_names]
    con_fecha = [(hoja, f) for hoja, f in con_fecha if f is not None]
    if not con_fecha:
        raise ValueError(
            f"No se encontró ninguna pestaña con formato de fecha en "
            f"{os.path.basename(ruta_excel)}. Pestañas disponibles: {xls.sheet_names}"
        )
    con_fecha.sort(key=lambda par: par[1])
    hoja_elegida, fecha_elegida = con_fecha[-1]
    if anio_esperado and mes_esperado and (fecha_elegida.year, fecha_elegida.month) != (anio_esperado, mes_esperado):
        print(
            f"  ⚠️ AVISO: la pestaña más reciente de {os.path.basename(ruta_excel)} es "
            f"'{hoja_elegida}' ({fecha_elegida}), pero se está procesando "
            f"{mes_esperado:02d}/{anio_esperado}. Es probable que esta fuente todavía no "
            f"tenga el corte de este mes — verifica antes de confiar en el resultado."
        )
    else:
        print(f"  usando pestaña '{hoja_elegida}' ({fecha_elegida}) de {os.path.basename(ruta_excel)}")
    return hoja_elegida


def leer_tabla_nominal(ruta_excel, hoja, columna_ancla='Número de trabajador'):
    """Lee una pestaña de planta sin asumir que el encabezado está en la fila 0.

    'Estructura de origen.txt' documenta que Abril, Junio y Julio traen una tabla
    resumen de 8 columnas ARRIBA del desglose nominal completo. Si no se detecta,
    esta función se comporta igual que un read_excel normal."""
    crudo = pd.read_excel(ruta_excel, sheet_name=hoja, header=None, dtype=str, nrows=15)
    fila_encabezado = None
    for i in range(len(crudo)):
        if crudo.iloc[i].astype(str).str.strip().eq(columna_ancla).any():
            fila_encabezado = i
            break
    if fila_encabezado is None:
        raise ValueError(
            f"No se encontró la columna '{columna_ancla}' en las primeras 15 filas de la "
            f"pestaña '{hoja}' de {os.path.basename(ruta_excel)}. Puede que el nombre de "
            f"columna haya cambiado; revisa el archivo a mano."
        )
    if fila_encabezado > 0:
        print(f"  aviso: la pestaña '{hoja}' trae {fila_encabezado} fila(s) de resumen "
              f"arriba de la tabla nominal; se usa la fila {fila_encabezado} como encabezado real.")
    return pd.read_excel(ruta_excel, sheet_name=hoja, header=fila_encabezado, dtype=str)


def parsear_fecha(valor):
    """Convierte serial de Excel, texto con distintos formatos o NaN a Timestamp."""
    if pd.isna(valor) or str(valor).strip() == '':
        return pd.NaT
    if isinstance(valor, (int, float)) and not isinstance(valor, bool):
        return pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(valor))
    texto = str(valor).strip()
    if re.fullmatch(r'\d+(\.\d+)?', texto):
        return pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(float(texto)))
    for fmt in ('%Y-%m-%d %H:%M:%S', '%Y-%m-%d', '%d/%m/%Y %H:%M:%S', '%d/%m/%Y', '%d-%m-%Y'):
        try:
            return pd.to_datetime(texto, format=fmt)
        except ValueError:
            continue
    return pd.to_datetime(texto, errors='coerce')


def texto_clave(valor):
    """Equivalente a la macro texto_clave() del SQL: sin acentos, mayúsculas, un solo espacio."""
    if pd.isna(valor):
        return ''
    texto = unicodedata.normalize('NFD', str(valor).strip())
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    return re.sub(r'\s+', ' ', texto).upper()


def meses_minimos(rango):
    """Convierte '0-3' o '3' en el número mínimo de meses (ver supuesto #2 arriba)."""
    texto = str(rango).strip()
    m = re.match(r'^\s*(\d+)\s*-\s*(\d+)\s*$', texto)
    if m:
        return int(m.group(1))
    m = re.match(r'^\s*(\d+)\s*$', texto)
    if m:
        return int(m.group(1))
    raise ValueError(f"No se pudo interpretar el rango de meses para cursar: '{rango}'")


def explotar_centros(df, columna='Centros de costos'):
    """Si una fila trae varios centros en una sola celda, la separa en varias filas."""
    df = df.copy()
    df[columna] = df[columna].astype(str).str.split(r'[,;\n/]+')
    df = df.explode(columna)
    df[columna] = df[columna].str.strip()
    return df[df[columna] != '']


def clave_unica(df, columna_llave, columna_valor):
    """Solo conserva llaves (persona/nombre/departamento) que apuntan a un único valor."""
    conteo = df.groupby(columna_llave)[columna_valor].nunique()
    llaves_unicas = conteo[conteo == 1].index
    return df[df[columna_llave].isin(llaves_unicas)].groupby(columna_llave)[columna_valor].min()


NOMBRES_HOJA_PDT = {
    'cursos_asignados': ['Cursos asignados', 'Cursos_asignados'],
    'colaboradores_asignados': ['Colaboradores asignados', 'Colaboradores_asignados'],
    'cursos_especificos': ['Cursos específicos', 'Cursos especificos', 'Cursos_especificos', 'Cursos_específicos'],
    'colaboradores_especificos': [
        'Colaboradores específicos', 'Colaboradores especificos',
        'Colaboradores_especificos', 'Colaboradores_específicos',
    ],
}


def cargar_plan_pdt_desde_xlsx(ruta_xlsx):
    """Lee las 4 pestañas de un PDT (Cursos/Colaboradores asignados y específicos)
    desde un único .xlsx, tolerando la variante sin acento en el nombre de pestaña."""
    xls = pd.ExcelFile(ruta_xlsx)

    def leer_hoja(nombres_posibles):
        for nombre in nombres_posibles:
            if nombre in xls.sheet_names:
                return limpiar_columnas(pd.read_excel(ruta_xlsx, sheet_name=nombre, dtype=str))
        raise ValueError(
            f"No se encontró ninguna pestaña entre {nombres_posibles} en "
            f"{os.path.basename(ruta_xlsx)}. Pestañas disponibles: {xls.sheet_names}"
        )

    return {clave: leer_hoja(nombres) for clave, nombres in NOMBRES_HOJA_PDT.items()}


def cargar_plan_pdt_desde_csv(rutas_csv):
    """Lee las 4 pestañas de un PDT desde 4 archivos .csv sueltos (uno por pestaña)."""
    return {clave: limpiar_columnas(pd.read_csv(ruta, dtype=str)) for clave, ruta in rutas_csv.items()}


def encontrar_plan_pdt(carpeta, alias, etiqueta):
    """Un PDT (plan de capacitación, Colaborador/Operación o Gerencial) puede
    llegar como 4 CSV sueltos (uno por pestaña, como
    'PDTgerencialadaptado08_26__Colaboradores_asignados.csv' o
    'PDT-operacion-adaptado-...-Cursos_asignados.csv') o como un solo .xlsx con
    las 4 pestañas. 'alias' es el patrón (estilo fnmatch, puede traer clases de
    caracteres como '[oó]' para variantes con/sin acento) que identifica al PDT
    dentro del nombre de archivo, sin importar el resto (guiones, fecha, etc.).
    Se prueban ambos formatos."""
    patrones_csv = {
        'cursos_asignados': f'*{alias}*Cursos_asignados*.csv',
        'colaboradores_asignados': f'*{alias}*Colaboradores_asignados*.csv',
        'cursos_especificos': f'*{alias}*Cursos_especificos*.csv',
        'colaboradores_especificos': f'*{alias}*Colaboradores_especificos*.csv',
    }
    rutas_csv = {clave: encontrar_archivo_opcional(carpeta, patron) for clave, patron in patrones_csv.items()}
    faltantes = [clave for clave, ruta in rutas_csv.items() if not ruta]

    if not faltantes:
        print(f"Plan {etiqueta}: encontrado como 4 CSV sueltos:")
        for clave, ruta in rutas_csv.items():
            print(f"   {clave}: {os.path.basename(ruta)}")
        return cargar_plan_pdt_desde_csv(rutas_csv)

    ruta_xlsx = encontrar_archivo_opcional(carpeta, f'*{alias}*.xlsx')
    if ruta_xlsx:
        print(f"Plan {etiqueta}: no se encontraron los 4 CSV sueltos (faltan: {faltantes}); "
              f"se usa el .xlsx {os.path.basename(ruta_xlsx)}.")
        return cargar_plan_pdt_desde_xlsx(ruta_xlsx)

    raise FileNotFoundError(
        f"No se encontró el plan de capacitación {etiqueta}: ni los 4 CSV sueltos "
        f"(faltan: {faltantes}) ni un .xlsx que combine con '*{alias}*', en {carpeta}."
    )


def procesar_plan_pdt(tablas, familia):
    """A partir de las 4 tablas crudas de un PDT, regresa las filas de plan ya
    explotadas (general + específico) con la columna plan_familia, para poder
    distinguir de qué archivo (Colaborador / Gerencial) salió cada asignación."""
    cursos_asignados = tablas['cursos_asignados'].copy()
    cursos_asignados['dias_minimos'] = cursos_asignados['Rango de meses para cursar'].apply(meses_minimos) * 30

    colaboradores_asignados = tablas['colaboradores_asignados'].copy()
    colaboradores_asignados['puesto_clave'] = colaboradores_asignados['Puesto'].apply(texto_clave)

    plan_general = colaboradores_asignados.merge(cursos_asignados, how='cross')
    plan_general = plan_general[['puesto_clave', 'Curso', 'ID Curso', 'dias_minimos']].copy()
    plan_general['centro_requerido'] = pd.NA
    plan_general['tipo_plan'] = 'General'

    cursos_especificos = tablas['cursos_especificos'].copy()
    cursos_especificos['dias_minimos'] = cursos_especificos['Rango de meses para cursar'].apply(meses_minimos) * 30

    colaboradores_especificos = explotar_centros(tablas['colaboradores_especificos'], 'Centros de costos')
    colaboradores_especificos['puesto_clave'] = colaboradores_especificos['Puesto'].apply(texto_clave)
    colaboradores_especificos['centro_requerido'] = (
        colaboradores_especificos['Centros de costos'].str.extract(r'(\d+)')[0]
    )

    plan_especifico = colaboradores_especificos.merge(cursos_especificos, how='cross')
    plan_especifico = plan_especifico[
        ['puesto_clave', 'centro_requerido', 'Curso', 'ID Curso', 'dias_minimos']
    ].copy()
    plan_especifico['tipo_plan'] = 'Específico'

    plan = pd.concat([plan_general, plan_especifico], ignore_index=True)
    plan = plan.rename(columns={'Curso': 'curso', 'ID Curso': 'curso_id'})
    plan['plan_familia'] = familia
    return plan


## 2 · Carga de fuentes crudas

In [5]:
ruta_detalle = encontrar_archivo(RUTA_CRUDOS, "*Detalle Colaborador*.xlsx")
ruta_posiciones = encontrar_archivo(RUTA_CRUDOS, "Planta de Cobranza por Posiciones*.xlsx")
ruta_centros = encontrar_archivo(RUTA_CRUDOS, "Planta de Cobranza por Centro*.xlsx")
rutas_periodos = encontrar_archivos_periodos(RUTA_CRUDOS)

print("Detalle colaborador:       ", os.path.basename(ruta_detalle))
print("Planta por posiciones:     ", os.path.basename(ruta_posiciones))
print("Planta por centro:         ", os.path.basename(ruta_centros))
print("Periodos de finalizaciones:", [os.path.basename(r) for r in rutas_periodos])


Detalle colaborador:        Cobranza_Detalle colaborador.xlsx
Planta por posiciones:      Planta de Cobranza por Posiciones_Cobranza.xlsx
Planta por centro:          Planta de Cobranza por Centro Autorizada_Activa.xlsx
Periodos de finalizaciones: ['Cobranza P1.csv', 'Cobranza P2.csv', 'Cobranza P3.csv']


In [6]:
# El PDT de Colaborador/Operación y el Gerencial usan el mismo formato de 4
# pestañas (Cursos/Colaboradores asignados + específicos) y pueden llegar como
# 4 CSV sueltos o como un .xlsx; solo cambia el alias que identifica a cada uno
# en el nombre de archivo (p. ej. "PDT-operacion-adaptado", "PDT-gerencial-adaptado").
tablas_plan_colaborador = encontrar_plan_pdt(RUTA_CRUDOS, 'operaci[oó]n', 'Colaborador (Operación)')
tablas_plan_gerencial = encontrar_plan_pdt(RUTA_CRUDOS, 'gerencial', 'Gerencial')

detalle = limpiar_columnas(pd.read_excel(ruta_detalle, sheet_name=0, dtype=str))

hoja_posiciones = hoja_mas_reciente(ruta_posiciones, AÑO, MES)
planta_posiciones = limpiar_columnas(leer_tabla_nominal(ruta_posiciones, hoja_posiciones))
centros_tipocentros = limpiar_columnas(pd.read_excel(ruta_posiciones, sheet_name="CENTROS-TIPOCENTROS", dtype=str))

hoja_centros = hoja_mas_reciente(ruta_centros, AÑO, MES)
planta_centros = limpiar_columnas(pd.read_excel(ruta_centros, sheet_name=hoja_centros, dtype=str))  # solo para auditoría

print(f"Detalle colaborador: {len(detalle):,} filas")
print(f"Planta por posiciones ({hoja_posiciones}): {len(planta_posiciones):,} filas")


Plan Colaborador (Operación): no se encontraron los 4 CSV sueltos (faltan: ['cursos_asignados', 'colaboradores_asignados', 'cursos_especificos', 'colaboradores_especificos']); se usa el .xlsx PDT-operacion-adaptado.xlsx.
Plan Gerencial: no se encontraron los 4 CSV sueltos (faltan: ['cursos_asignados', 'colaboradores_asignados', 'cursos_especificos', 'colaboradores_especificos']); se usa el .xlsx PDT-gerencial-adaptado.xlsx.
  usando pestaña '01 AGO 26' (2026-08-01) de Planta de Cobranza por Posiciones_Cobranza.xlsx
  usando pestaña '01 AGO 26' (2026-08-01) de Planta de Cobranza por Centro Autorizada_Activa.xlsx
Detalle colaborador: 122,783 filas
Planta por posiciones (01 AGO 26): 12,903 filas


## 3 · Plan de capacitación (Colaborador + Gerencial, general + específico)

Regla general: cada puesto de `Colaboradores asignados` recibe todos los cursos de
`Cursos asignados`. Regla específica: cada combinación puesto+centro de
`Colaboradores específicos` recibe los cursos de `Cursos específicos`.

El plan final **une** las asignaciones de los dos archivos (Colaborador y Gerencial).
Un puesto que aparezca en ambos —como 721 (Gerente de Operación Cobranza) o 743 (Jefe
de Operación Cobranza), que están tanto en la especialización de conductores del plan
de Colaborador como en la malla del plan Gerencial— recibe los cursos de **los dos**,
tal como confirmó el equipo de negocio.

In [7]:
plan_colaborador = procesar_plan_pdt(tablas_plan_colaborador, 'Colaborador')
plan_gerencial = procesar_plan_pdt(tablas_plan_gerencial, 'Gerencial')

plan_cobranza = pd.concat([plan_colaborador, plan_gerencial], ignore_index=True)
plan_cobranza.insert(0, 'plan_orden', range(len(plan_cobranza)))

print(f"Plan de capacitación total: {len(plan_cobranza):,} filas")
print(plan_cobranza.groupby(['plan_familia', 'tipo_plan']).size().rename('filas').to_string())
plan_cobranza.head()


ValueError: No se pudo interpretar el rango de meses para cursar: '2024-06-03 00:00:00'

## 4 · Centro y Región de Cobranza

Mismo criterio en cascada que el SQL viejo (número de persona → nombre único →
departamento único y sin contradicciones), usando la Planta por Posiciones más
reciente. La Región de Cobranza sale de cruzar el Centro asignado contra
`CENTROS-TIPOCENTROS` (la "Plantilla de Cobranza" del PDF), no de la región cruda de
Detalle Colaborador.

In [ ]:
planta_posiciones = planta_posiciones.copy()
planta_posiciones['numero_persona'] = pd.to_numeric(
    planta_posiciones['Número de trabajador'], errors='coerce'
).astype('Int64')
planta_posiciones['nombre_clave'] = planta_posiciones['Nombre del colaborador'].apply(texto_clave)
planta_posiciones['departamento_clave'] = planta_posiciones['Departamento'].apply(texto_clave)
planta_posiciones['centro_cobranza'] = pd.to_numeric(
    planta_posiciones['Centro'], errors='coerce'
).astype('Int64')
planta_posiciones = planta_posiciones.dropna(subset=['centro_cobranza'])

centro_por_persona = clave_unica(
    planta_posiciones.dropna(subset=['numero_persona']), 'numero_persona', 'centro_cobranza'
)
centro_por_nombre = clave_unica(
    planta_posiciones[planta_posiciones['nombre_clave'] != ''], 'nombre_clave', 'centro_cobranza'
)
centro_por_departamento = clave_unica(
    planta_posiciones[planta_posiciones['departamento_clave'] != ''],
    'departamento_clave', 'centro_cobranza',
)

# Descarta departamentos cuyo centro deducido contradiga el centro individual de
# alguna persona (igual que 'departamentos_conflictivos_cobranza' en el SQL viejo).
personas_centro = detalle.copy()
personas_centro['numero_persona'] = pd.to_numeric(
    personas_centro['Número de persona'], errors='coerce'
).astype('Int64')
personas_centro['departamento_clave'] = personas_centro['Nombre del departamento'].apply(texto_clave)
comparacion = personas_centro.merge(
    centro_por_persona.rename('centro_por_persona'), left_on='numero_persona', right_index=True
).merge(
    centro_por_departamento.rename('centro_por_departamento'), left_on='departamento_clave', right_index=True
)
departamentos_conflictivos = set(
    comparacion.loc[
        comparacion['centro_por_persona'] != comparacion['centro_por_departamento'], 'departamento_clave'
    ]
)
centro_por_departamento_seguro = centro_por_departamento.drop(
    index=[k for k in departamentos_conflictivos if k in centro_por_departamento.index]
)

print(f"Departamentos descartados por contradecir un Centro individual conocido: "
      f"{len(departamentos_conflictivos)}")


In [ ]:
detalle = detalle.copy()
detalle['numero_persona'] = pd.to_numeric(detalle['Número de persona'], errors='coerce').astype('Int64')
detalle['nombre_clave'] = detalle['Nombre'].apply(texto_clave)
detalle['departamento_clave'] = detalle['Nombre del departamento'].apply(texto_clave)
detalle['puesto_clave'] = detalle['Nombre de puesto'].apply(texto_clave)
detalle['fecha_contratacion'] = detalle['Fecha de contratación de la empresa'].apply(parsear_fecha)

detalle['centro_cobranza'] = detalle['numero_persona'].map(centro_por_persona)
detalle['origen_centro'] = np.where(detalle['centro_cobranza'].notna(), 'Número de persona', pd.NA)

falta = detalle['centro_cobranza'].isna()
detalle.loc[falta, 'centro_cobranza'] = detalle.loc[falta, 'nombre_clave'].map(centro_por_nombre)
detalle.loc[falta & detalle['centro_cobranza'].notna(), 'origen_centro'] = 'Nombre único'

falta = detalle['centro_cobranza'].isna()
detalle.loc[falta, 'centro_cobranza'] = (
    detalle.loc[falta, 'departamento_clave'].map(centro_por_departamento_seguro)
)
detalle.loc[falta & detalle['centro_cobranza'].notna(), 'origen_centro'] = 'Departamento único'

detalle['origen_centro'] = detalle['origen_centro'].fillna('Pendiente')

print("Cobertura de Centro:")
print(detalle['origen_centro'].value_counts())


In [ ]:
centros_tipocentros = centros_tipocentros.copy()
centros_tipocentros['centro_cobranza'] = pd.to_numeric(
    centros_tipocentros['# Centro'], errors='coerce'
).astype('Int64')
mapa_region_cobranza = (
    centros_tipocentros.dropna(subset=['centro_cobranza'])
    .drop_duplicates('centro_cobranza')
    .set_index('centro_cobranza')['REGION COBRANZA']
)

detalle['region_cobranza'] = detalle['centro_cobranza'].map(mapa_region_cobranza)

regiones_no_reconocidas = (
    detalle.loc[detalle['region_cobranza'].notna(), 'region_cobranza']
    .apply(texto_clave)
    .loc[lambda s: ~s.isin(REGIONES_COBRANZA_OFICIALES)]
    .value_counts()
)
print("Región Cobranza fuera de las 15 oficiales (revisar catálogo si no está vacío):")
print(regiones_no_reconocidas if len(regiones_no_reconocidas) else "  (ninguna) ✅")


## 5 · Persona + puesto → cursos que aplican

In [ ]:
fecha_corte_ts = pd.Timestamp(FECHA_CORTE)
detalle['dias_laborados'] = (fecha_corte_ts - detalle['fecha_contratacion']).dt.days
detalle_valido = detalle[detalle['dias_laborados'] >= 0].copy()

descartados = len(detalle) - len(detalle_valido)
if descartados:
    print(f"Aviso: {descartados} filas con fecha de contratación posterior al corte se descartaron.")

aplican = detalle_valido.merge(plan_cobranza, on='puesto_clave', how='inner')
aplican = aplican[
    aplican['centro_requerido'].isna()
    | (aplican['centro_requerido'] == aplican['centro_cobranza'].astype('Int64').astype(str))
]
aplican = aplican[aplican['dias_laborados'] >= aplican['dias_minimos']].copy()

print(f"Asignaciones curso-persona antes de cruzar finalizaciones: {len(aplican):,}")

# Puestos de Detalle que nunca calzan con el plan (para revisar cobertura; no es
# necesariamente un error, ya que no todo puesto de Cobranza está en el plan).
puestos_sin_plan = (
    detalle_valido.loc[~detalle_valido['puesto_clave'].isin(plan_cobranza['puesto_clave']), 'Nombre de puesto']
    .value_counts()
)
print("\nPuestos de Detalle Colaborador sin ningún curso asignado en el plan:")
print(puestos_sin_plan if len(puestos_sin_plan) else "  (ninguno)")


## 6 · Cruce con finalizaciones (sin deduplicar)

Se concentran todos los `Cobranza_P*.csv` disponibles. Los duplicados persona+curso
**no se eliminan**: igual que en el SQL viejo, cada duplicado suma tanto a Total como
a Completados.

In [ ]:
partes = []
for ruta in rutas_periodos:
    m = re.search(r'P(\d+)', os.path.basename(ruta), flags=re.IGNORECASE)
    periodo_id = f"P{m.group(1)}" if m else os.path.splitext(os.path.basename(ruta))[0]
    parte = pd.read_csv(ruta, dtype=str)
    parte['periodo'] = periodo_id
    partes.append(parte)
    print(f"  {os.path.basename(ruta)} -> periodo '{periodo_id}', {len(parte):,} filas")

cobranza_p = pd.concat(partes, ignore_index=True)

finalizados = pd.DataFrame({
    'numero_persona': pd.to_numeric(cobranza_p['Número Persona'], errors='coerce').astype('Int64'),
    'curso': cobranza_p['Nombre Curso'].astype(str).str.strip(),
    'lo_completo': cobranza_p['¿Lo Completó?'].astype(str).str.strip().str.lower(),
    'fecha_finalizado': cobranza_p['Fecha Finalizado'],
    'periodo': cobranza_p['periodo'],
})

resultados = aplican.merge(finalizados, on=['numero_persona', 'curso'], how='left')
resultados['completados'] = resultados['lo_completo'].isin(['si', 'sí']).astype(int)
resultados['total'] = 1
resultados['pendiente'] = resultados['completados'] == 0

print(f"\nFilas de resultado (una por curso asignado, con o sin finalización): {len(resultados):,}")

# Cursos del plan que jamás encontraron ninguna finalización que calzara por nombre
# (posible desajuste de nombre entre el plan y el csv de finalizaciones; ver supuesto #3).
cursos_sin_ninguna_coincidencia = sorted(
    set(plan_cobranza['curso']) - set(finalizados['curso'])
)
print("\nCursos del plan que nunca aparecen en Cobranza_P*.csv (revisar nombre exacto):")
for c in cursos_sin_ninguna_coincidencia:
    print(f"  - {c}")
if not cursos_sin_ninguna_coincidencia:
    print("  (ninguno) ✅")


## 7 · Validaciones

In [ ]:
# --- Excepción Centro de Impresión de Cobranza (PDF, página 2) -------------------
# Solo aplica a la especialización de conductores del plan de COLABORADOR; el plan
# Gerencial es un paquete aparte que sí les toca a 721/743 sin importar el centro.
sospechosos = aplican[
    (aplican['plan_familia'] == 'Colaborador')
    & (aplican['tipo_plan'] == 'Específico')
    & (aplican['Código de puesto'].astype(str).str.extract(r'(\d+)')[0].isin(PUESTOS_EXCEPCION_IMPRESION))
    & (aplican['centro_cobranza'].astype('Int64').astype(str).isin(CENTROS_IMPRESION_COBRANZA))
]
print("Personas con puesto 743/721 en un centro de Impresión de Cobranza que de "
      "todas formas recibieron la especialización de conductores (debería ser 0):")
print(len(sospechosos))
if len(sospechosos):
    display(sospechosos[['numero_persona', 'Nombre', 'Código de puesto', 'centro_cobranza', 'curso']])


In [ ]:
# --- Personas que reciben cursos de AMBOS planes (Colaborador + Gerencial) -------
# Es el caso esperado para 721/743 (Gerente/Jefe de Operación Cobranza): confirma
# que la regla de "se suman" quedó bien aplicada, no que algo se está duplicando.
planes_por_persona = aplican.groupby('numero_persona')['plan_familia'].apply(lambda s: set(s))
personas_ambos_planes = planes_por_persona[planes_por_persona.apply(len) > 1]
print(f"Personas que reciben cursos de los dos planes a la vez: {len(personas_ambos_planes)}")
if len(personas_ambos_planes):
    ejemplo = aplican[aplican['numero_persona'].isin(personas_ambos_planes.index[:5])]
    display(ejemplo[['numero_persona', 'Nombre', 'Código de puesto', 'plan_familia', 'curso']]
            .sort_values(['numero_persona', 'plan_familia']))


In [ ]:
# --- Control algebraico: ambas salidas deben sumar exactamente lo mismo ----------
print("Total / Completados agregados en 'resultados' (deben coincidir con las 2 salidas):")
print(resultados[['total', 'completados']].sum())


## 8 · Salidas finales

Mismas columnas que `Tuberia/Estructura de resultado.txt`, agregando **Centro**
(que ese ejemplo no trae porque esa área no lo usa, pero Cobranza sí lo necesita).

In [ ]:
def unir_pendientes(grupo):
    pendientes = grupo.loc[grupo['pendiente']].sort_values('plan_orden')
    return ', '.join(pendientes['curso'])


detalle_colaborador = (
    resultados.groupby(
        ['numero_persona', 'Nombre', 'fecha_contratacion', 'Código de puesto', 'Nombre de puesto',
         'region_cobranza', 'Nombre del departamento', 'centro_cobranza', 'dias_laborados'],
        dropna=False,
    )
    .apply(lambda g: pd.Series({
        'Total': g['total'].sum(),
        'Completados': g['completados'].sum(),
        'Pendientes': unir_pendientes(g),
    }))
    .reset_index()
)

detalle_colaborador['Centro'] = detalle_colaborador['centro_cobranza'].apply(
    lambda c: 'Pendiente' if pd.isna(c) else str(int(c))
)
detalle_colaborador['Fecha corte'] = FECHA_CORTE

detalle_colaborador = detalle_colaborador.rename(columns={
    'numero_persona': 'Número de persona',
    'Nombre': 'Nombre',
    'fecha_contratacion': 'Fecha de contratación de la empresa',
    'Código de puesto': 'Código de puesto',
    'Nombre de puesto': 'Nombre de puesto',
    'region_cobranza': 'Región',
    'Nombre del departamento': 'Nombre del departamento',
    'dias_laborados': 'Dias laborados',
})[[
    'Número de persona', 'Nombre', 'Fecha de contratación de la empresa', 'Código de puesto',
    'Nombre de puesto', 'Región', 'Nombre del departamento', 'Centro', 'Fecha corte',
    'Dias laborados', 'Total', 'Completados', 'Pendientes',
]]

print(f"Detalle Colaborador: {len(detalle_colaborador):,} filas")
detalle_colaborador.head()


In [ ]:
avance_curso_region = (
    resultados.groupby(['curso', 'region_cobranza', 'Nombre del departamento', 'centro_cobranza'], dropna=False)
    .agg(Total=('total', 'sum'), Completados=('completados', 'sum'))
    .reset_index()
)
avance_curso_region['Centro'] = avance_curso_region['centro_cobranza'].apply(
    lambda c: 'Pendiente' if pd.isna(c) else str(int(c))
)
avance_curso_region = avance_curso_region.rename(columns={
    'curso': 'Curso',
    'region_cobranza': 'Región',
}).drop(columns=['centro_cobranza'])[
    ['Curso', 'Región', 'Nombre del departamento', 'Centro', 'Total', 'Completados']
]

print(f"Avance por curso - región - centro: {len(avance_curso_region):,} filas")

# Control algebraico final (debe imprimir números idénticos en cada columna).
print("\nControl algebraico:")
print(f"  Total:       detalle={detalle_colaborador['Total'].sum():,}  "
      f"avance={avance_curso_region['Total'].sum():,}")
print(f"  Completados: detalle={detalle_colaborador['Completados'].sum():,}  "
      f"avance={avance_curso_region['Completados'].sum():,}")

avance_curso_region.head()


## 9 · Exportar CSV a `Resultados`

In [ ]:
os.makedirs(RUTA_RESULTADOS, exist_ok=True)

ruta_detalle_csv = os.path.join(RUTA_RESULTADOS, f"Detalle Colaborador Cobranza {ETIQUETA_MES}.csv")
ruta_avance_csv = os.path.join(RUTA_RESULTADOS, f"Avance Curso Región Centro Cobranza {ETIQUETA_MES}.csv")

# utf-8-sig para que los acentos se vean bien al abrir el CSV directo en Excel.
detalle_colaborador.to_csv(ruta_detalle_csv, index=False, encoding='utf-8-sig')
avance_curso_region.to_csv(ruta_avance_csv, index=False, encoding='utf-8-sig')

print("Exportado:")
print(" ", ruta_detalle_csv)
print(" ", ruta_avance_csv)


## 10 · Exportar paquete para el tablero (`.macintosh.json`)

Los dos CSV de arriba son para lectura humana (Excel). El tablero de Cobranza
(Apps Script, carpeta `src/` del repo) no lee esos CSV directamente: se alimenta
subiendo un archivo `.macintosh.json` desde su botón **"Actualizar datos"**. Esta
sección arma ese paquete a partir de los mismos DataFrames que ya calculamos arriba
(no vuelve a leer los CSV), con las 5 pestañas que el tablero espera: `Resumen`,
`Regiones`, `Cursos`, `Puestos` y `Colaboradores`.

Nota: el tablero usa la palabra **Centro** (no "Tienda", que era el término del
tablero de Almacenista en el que se basó el diseño) — eso ya quedó corregido en
`src/JavaScript.html` y `src/DataService.gs`.

In [ ]:
def limpiar(valor):
    """None/NaN -> None; numpy int/float -> tipos nativos de Python (para que json.dump no truene)."""
    if valor is None:
        return None
    try:
        if pd.isna(valor):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(valor, (np.integer,)):
        return int(valor)
    if isinstance(valor, (np.floating,)):
        return float(valor)
    return valor


def fila_limpia(valores):
    return [limpiar(v) for v in valores]


# --- Resumen: una sola fila con los totales generales -----------------------
total_colaboradores = int(detalle_colaborador['Número de persona'].nunique())
total_asignados = int(resultados['total'].sum())
total_completados = int(resultados['completados'].sum())
total_pendientes = total_asignados - total_completados
avance_general = (total_completados / total_asignados) if total_asignados else 0

resumen_rows = [fila_limpia([
    'Cobranza', ETIQUETA_MES, FECHA_CORTE,
    total_colaboradores, total_asignados, total_completados, total_pendientes, avance_general,
])]

# --- Regiones: por persona (no por curso), para que 'colaboradores' cuente personas únicas ---
regiones_rows = []
for region, grupo in detalle_colaborador.groupby('Región', dropna=False):
    asignados = int(grupo['Total'].sum())
    completados = int(grupo['Completados'].sum())
    regiones_rows.append(fila_limpia([
        region, int(grupo['Número de persona'].nunique()), asignados, completados,
        asignados - completados, (completados / asignados) if asignados else 0,
    ]))

# --- Cursos: asignados/completados por curso, sumando todas las regiones/centros ---
cursos_rows = []
for curso, grupo in resultados.groupby('curso', dropna=False):
    asignados = int(grupo['total'].sum())
    completados = int(grupo['completados'].sum())
    # curso_clave se deja vacío a propósito: el tablero lo calcula solo (_slug)
    # a partir del nombre si no se lo mandamos, y así evitamos que dos
    # implementaciones distintas del "slug" (Python vs JS) puedan desalinearse.
    cursos_rows.append(fila_limpia([
        curso, '', asignados, completados, (completados / asignados) if asignados else 0,
    ]))

# --- Puestos: catálogo de puestos distintos ---------------------------------
puestos_rows = [[p] for p in sorted(detalle_colaborador['Nombre de puesto'].dropna().unique())]

# --- Colaboradores: una fila por persona, cursos separados por ' | ' --------
resultados_por_persona = resultados.groupby('numero_persona')
colaboradores_rows = []
for _, fila in detalle_colaborador.iterrows():
    persona_id = fila['Número de persona']
    if persona_id in resultados_por_persona.groups:
        cursos_persona = resultados_por_persona.get_group(persona_id)
        lista_asignados = ' | '.join(cursos_persona['curso'].tolist())
        lista_pendientes = ' | '.join(cursos_persona.loc[cursos_persona['pendiente'], 'curso'].tolist())
    else:
        lista_asignados = lista_pendientes = ''
    colaboradores_rows.append(fila_limpia([
        fila['Nombre'], str(persona_id), fila['Centro'], fila['Nombre de puesto'], fila['Región'],
        int(fila['Total']), int(fila['Completados']), lista_pendientes, lista_asignados,
    ]))

paquete_tablero = {
    'format': 'macintosh-report-package',
    'version': 1,
    'report': 'cobranza',
    'period': FECHA_CORTE[:7],
    'cutoffDate': FECHA_CORTE,
    'sheets': {
        'Resumen': {'rows': resumen_rows},
        'Regiones': {'rows': regiones_rows},
        'Cursos': {'rows': cursos_rows},
        'Puestos': {'rows': puestos_rows},
        'Colaboradores': {'rows': colaboradores_rows},
    },
}

ruta_paquete = os.path.join(RUTA_RESULTADOS, "cobranza.macintosh.json")
with open(ruta_paquete, 'w', encoding='utf-8') as f:
    json.dump(paquete_tablero, f, ensure_ascii=False, allow_nan=False)

print(f"Paquete para el tablero: {len(colaboradores_rows):,} colaboradores, "
      f"{len(regiones_rows)} regiones, {len(cursos_rows)} cursos, {len(puestos_rows)} puestos.")
print("Exportado en:", ruta_paquete)
print("\nSúbelo en el tablero con el botón 'Actualizar datos' (icono ⚙️ arriba a la derecha del reporte).")
